# Vision Transformer（ViT）在 MNIST 上的原理詳解
# How Vision Transformer Works on MNIST

---

## 核心概念 / Core Idea

> **中文：** 把圖像切成小塊（patch），每塊當作一個「詞元（token）」，然後用 Transformer 處理這個詞元序列。  
> **EN:** Split an image into patches, treat each patch like a word token, and process the sequence with a Transformer.

```
圖像 (28×28)  →  切成小塊 (16塊)  →  每塊線性嵌入  →  Transformer  →  分類結果
Image          Split into patches    Embed each patch   Transformer     Class label
```

**本筆記內容 / Contents:**
1. 視覺領域怎麼應用這個架構 / Vision Domain Applications
2. 資料載入與視覺化 / Data Loading & Patch Visualization
3. 塊嵌入 / Patch Embedding
4. 位置編碼 / Positional Encoding
5. 多頭自注意力 / Multi-Head Self-Attention
6. Transformer 塊 / Transformer Block
7. 完整 MNIST 分類模型 / Full MNIST Classification Model
8. 訓練與評估 / Training & Evaluation
9. 混淆矩陣與各類準確率 / Confusion Matrix & Per-class Accuracy
10. 注意力視覺化 / Attention Visualization
11. 注意力傳播 / Attention Rollout
12. 與 vit-pytorch 比較 / Compare with vit-pytorch

## 環境安裝 / Setup

In [ ]:
!pip install einops vit-pytorch --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from einops import rearrange, repeat
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from sklearn.metrics import confusion_matrix
import itertools

# 支援繁體中文字型顯示 / Enable Traditional Chinese font
plt.rcParams['font.family'] = ['Microsoft JhengHei', 'MingLiU', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'使用裝置 / Device: {device}')
print(f'PyTorch 版本 / Version: {torch.__version__}')

---
## 第一步：視覺領域怎麼應用這個架構 / Step 1 — Vision Domain Applications

**中文：**  
「視覺領域（Vision Domain）」是指所有以**圖像或影片像素**為輸入的任務。  
在 2020 年 ViT 出現之前，視覺領域幾乎完全由 CNN 主導。ViT 證明了 Transformer 架構同樣適用於視覺任務，甚至在大規模資料下超越 CNN。

**EN:**  
The "vision domain" covers all tasks where input data is images or video pixels.  
Before ViT (2020), vision was dominated by CNNs. ViT proved Transformers work equally well — and at scale, even better.

---

### 主要任務類型 / Core Task Types

| 任務 Task | 輸出 Output | ViT 如何應用 How ViT applies | 例子 Example |
|---|---|---|---|
| **圖像分類** Image Classification | 單一標籤 One label | CLS 詞元輸出接分類頭 | MNIST、ImageNet |
| **目標偵測** Object Detection | 邊界框 + 標籤 | Patch 詞元接偵測頭（DETR） | 行人偵測 |
| **語意分割** Semantic Segmentation | 每像素標籤 | Patch 嵌入接上採樣頭（SegFormer） | 自駕車道路分析 |
| **醫療影像** Medical Imaging | 病灶分類/分割 | MAE 預訓練 + 微調 | X 光、病理切片 |
| **影片理解** Video Understanding | 動作標籤 | ViT-3D：加入時間軸塊 | 動作辨識 |
| **多模態** Multimodal | 文字/圖像對齊 | ViT 作為視覺編碼器（CLIP） | 圖文搜尋 |
| **圖像生成** Image Generation | 新圖像 | DiT：Transformer 替換 U-Net | Stable Diffusion |

---

### 架構如何對應各任務 / How the Architecture Maps to Each Task

```
同一個 ViT 主體（Backbone）
         │
         ├─→ CLS 詞元輸出 ──→ Linear ──→ 【圖像分類】
         │
         ├─→ 所有 Patch 詞元 ──→ 偵測頭 ──→ 【目標偵測】
         │
         ├─→ 所有 Patch 詞元 ──→ 上採樣 ──→ 【語意分割】
         │
         └─→ 特徵向量 ──→ 跨模態注意力 ──→ 【多模態】
```

> **本筆記專注於最基礎的任務：圖像分類，以 MNIST 手寫數字為例。**  
> This notebook focuses on the most fundamental task: image classification, using MNIST handwritten digits.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左圖：任務類型 / Left: task types
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Transformer 架構在視覺領域的應用\nTransformer Architecture Applications in Vision Domain', fontsize=11, pad=10)

backbone = mpatches.FancyBboxPatch((3.5, 4.0), 3.0, 2.0,
    boxstyle='round,pad=0.15', facecolor='#AED6F1', edgecolor='#2E86C1', linewidth=2)
ax.add_patch(backbone)
ax.text(5.0, 5.0, 'ViT\nBackbone', ha='center', va='center', fontsize=11, fontweight='bold')

tasks = [
    (1.0, 8.5, '#A9DFBF', '圖像分類\nClassification', '#1E8449'),
    (1.0, 6.5, '#F9E79F', '目標偵測\nDetection',     '#B7950B'),
    (1.0, 4.5, '#FAD7A0', '語意分割\nSegmentation',  '#D35400'),
    (7.0, 8.5, '#D2B4DE', '醫療影像\nMedical',       '#7D3C98'),
    (7.0, 6.5, '#F1948A', '影片理解\nVideo',         '#C0392B'),
    (7.0, 4.5, '#AED6F1', '多模態\nMultimodal',      '#1A5276'),
]
for (tx, ty, fc, label, ec) in tasks:
    b = mpatches.FancyBboxPatch((tx, ty-0.6), 2.2, 1.2,
        boxstyle='round,pad=0.1', facecolor=fc, edgecolor=ec, linewidth=1.5)
    ax.add_patch(b)
    ax.text(tx+1.1, ty, label, ha='center', va='center', fontsize=8, fontweight='bold')

# Arrows from backbone
arrow_targets = [(2.2, 8.5), (2.2, 6.5), (2.2, 4.5), (7.0, 8.5), (7.0, 6.5), (7.0, 4.5)]
arrow_starts  = [(3.5, 5.5), (3.5, 5.0), (3.5, 4.5), (6.5, 5.5), (6.5, 5.0), (6.5, 4.5)]
for (sx, sy), (ex, ey) in zip(arrow_starts, arrow_targets):
    ax.annotate('', xy=(ex, ey), xytext=(sx, sy),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5,
                                connectionstyle='arc3,rad=0.1'))

# 右圖：CNN vs ViT 感受野對比 / Right: CNN vs ViT receptive field
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.axis('off')
ax2.set_title('CNN vs ViT 感受野差異\nCNN vs ViT Receptive Field', fontsize=11, pad=10)

# CNN side
ax2.text(1.5, 9.2, 'CNN', ha='center', fontsize=10, fontweight='bold', color='#C0392B')
for layer, size, color in [(0, 2.0, '#FADBD8'), (1, 2.8, '#F1948A'), (2, 3.6, '#E74C3C')]:
    b = mpatches.FancyBboxPatch((1.5 - size/2, 7.5 - layer*1.8 - size/2), size, size,
        boxstyle='round,pad=0.05', facecolor=color, edgecolor='#922B21', linewidth=1, alpha=0.8)
    ax2.add_patch(b)
    ax2.text(1.5, 7.5 - layer*1.8, f'Layer {layer+1}\n局部感受野', ha='center', va='center', fontsize=7)

ax2.annotate('', xy=(1.5, 2.2), xytext=(1.5, 3.0),
             arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))
ax2.text(1.5, 1.8, '需多層才能\n看到全圖', ha='center', fontsize=8, color='#C0392B')

# ViT side
ax2.text(7.5, 9.2, 'ViT', ha='center', fontsize=10, fontweight='bold', color='#1A5276')
grid_origin = (5.5, 5.5)
colors_grid = plt.cm.Blues(np.linspace(0.3, 0.9, 16)).reshape(4, 4, 4)
for i in range(4):
    for j in range(4):
        b = mpatches.Rectangle((grid_origin[0] + j*0.95, grid_origin[1] + (3-i)*0.95),
            0.9, 0.9, facecolor=colors_grid[i,j], edgecolor='white', linewidth=1)
        ax2.add_patch(b)
        ax2.text(grid_origin[0]+j*0.95+0.45, grid_origin[1]+(3-i)*0.95+0.45,
                 f'{i*4+j}', ha='center', va='center', fontsize=6, color='white')

# Show attention lines from center patch to all others
center = (grid_origin[0]+1.5*0.95+0.45, grid_origin[1]+1.5*0.95+0.45)
for i in range(4):
    for j in range(4):
        if not (i == 1 and j == 1):
            target = (grid_origin[0]+j*0.95+0.45, grid_origin[1]+(3-i)*0.95+0.45)
            ax2.plot([center[0], target[0]], [center[1], target[1]],
                     color='#E74C3C', alpha=0.3, linewidth=0.8)

ax2.text(7.5, 4.8, '第1層即可看到全圖\nLayer 1: global view', ha='center', fontsize=8, color='#1A5276')
ax2.text(7.5, 4.2, '每個塊直接注意所有其他塊', ha='center', fontsize=7, color='#555')

plt.tight_layout()
plt.show()

---
## 第二步：載入 MNIST 資料 / Step 2 — Load MNIST

**中文：**  
MNIST 是 28×28 的灰階手寫數字圖像，共 10 類（0–9）。  
我們選擇 **7×7 的塊大小**，這樣每張圖像被切成 **4×4 = 16 塊**。

**EN:**  
MNIST is 28×28 grayscale images with 10 classes (digits 0–9).  
We use **patch size 7×7**, giving a **4×4 grid = 16 patches** per image.

In [ ]:
IMAGE_SIZE  = 28
PATCH_SIZE  = 7
NUM_PATCHES = (IMAGE_SIZE // PATCH_SIZE) ** 2  # 4×4 = 16
PATCH_DIM   = PATCH_SIZE * PATCH_SIZE * 1      # 7×7×1 = 49

print('── 關鍵參數 / Key Parameters ──')
print(f'圖像尺寸   Image size   : {IMAGE_SIZE}×{IMAGE_SIZE}')
print(f'塊尺寸     Patch size   : {PATCH_SIZE}×{PATCH_SIZE}')
print(f'塊數量     Num patches  : {NUM_PATCHES}  （網格 {IMAGE_SIZE//PATCH_SIZE}×{IMAGE_SIZE//PATCH_SIZE}）')
print(f'塊維度     Patch dim    : {PATCH_DIM}  （每塊像素數）')

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_data = datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_data  = datasets.MNIST('./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_data,  batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

print(f'\n訓練集 Train: {len(train_data):,} 張')
print(f'測試集 Test:  {len(test_data):,} 張')

### 視覺化圖像塊 / Visualize Patches

**中文：** 每個彩色方格就是 Transformer 將要處理的一個「詞元（token）」。  
**EN:** Each colored cell is one token the Transformer will process.

In [ ]:
sample_img, sample_label = train_data[0]
img_np = sample_img.squeeze().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img_np, cmap='gray')
axes[0].set_title(f'原始圖像 / Original\n標籤 Label: {sample_label}', fontsize=12)
axes[0].axis('off')

axes[1].imshow(img_np, cmap='gray')
n = IMAGE_SIZE // PATCH_SIZE
colors = plt.cm.tab20(np.linspace(0, 1, NUM_PATCHES))
for i in range(n):
    for j in range(n):
        idx = i * n + j
        rect = mpatches.Rectangle(
            (j * PATCH_SIZE - 0.5, i * PATCH_SIZE - 0.5),
            PATCH_SIZE, PATCH_SIZE,
            linewidth=2, edgecolor=colors[idx], facecolor='none'
        )
        axes[1].add_patch(rect)
        axes[1].text(
            j * PATCH_SIZE + PATCH_SIZE // 2 - 0.5,
            i * PATCH_SIZE + PATCH_SIZE // 2,
            str(idx), color=colors[idx], fontsize=9, fontweight='bold', ha='center'
        )
axes[1].set_title(f'切分為 {NUM_PATCHES} 個塊 / {NUM_PATCHES} patches\n每塊 = 1 個 token', fontsize=11)
axes[1].axis('off')

img_tensor = sample_img.unsqueeze(0)
patches_t = rearrange(img_tensor, 'b c (h p1) (w p2) -> b (h w) (p1 p2 c)',
                       p1=PATCH_SIZE, p2=PATCH_SIZE)
patch_display = patches_t[0].numpy().reshape(NUM_PATCHES, PATCH_SIZE, PATCH_SIZE)
combined = np.zeros((4 * PATCH_SIZE, 4 * PATCH_SIZE))
for idx in range(NUM_PATCHES):
    r, c = idx // 4, idx % 4
    combined[r*PATCH_SIZE:(r+1)*PATCH_SIZE, c*PATCH_SIZE:(c+1)*PATCH_SIZE] = patch_display[idx]
axes[2].imshow(combined, cmap='gray')
for i in range(1, 4):
    axes[2].axhline(i * PATCH_SIZE - 0.5, color='red', linewidth=1.5)
    axes[2].axvline(i * PATCH_SIZE - 0.5, color='red', linewidth=1.5)
axes[2].set_title(f'提取的 {NUM_PATCHES} 個塊\n每塊展平為 {PATCH_DIM} 維向量', fontsize=11)
axes[2].axis('off')

plt.suptitle('ViT 第一步：把圖像變成詞元序列 / Convert Image into Token Sequence', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print(f'塊張量形狀: {patches_t.shape}  (batch=1, 塊數=16, 維度=49)')

---
## 第三步：塊嵌入 / Step 3 — Patch Embedding

**中文：**  
每個塊是 49 個原始像素值。用一個線性層將其映射到更高維度（`dim`），就像 NLP 中的詞嵌入。  
在序列開頭加入可學習的 **[CLS] 詞元**，Transformer 結束後只用它的輸出做分類。

**EN:**  
A linear layer projects each 49-pixel patch to `dim` dimensions — like a word embedding.  
A learnable **[CLS] token** is prepended; only its output is used for classification.

```
塊 patch（49維）  →  Linear(49, dim)  →  嵌入 embedding（dim維）
```

In [ ]:
class PatchEmbedding(nn.Module):
    """將圖像切塊並線性投影為詞元序列。"""
    def __init__(self, patch_dim, dim):
        super().__init__()
        self.projection = nn.Linear(patch_dim, dim)
        self.cls_token  = nn.Parameter(torch.randn(1, 1, dim))

    def forward(self, x):
        patches = rearrange(x, 'b c (h p1) (w p2) -> b (h w) (p1 p2 c)',
                             p1=PATCH_SIZE, p2=PATCH_SIZE)
        tokens = self.projection(patches)
        cls    = repeat(self.cls_token, '1 1 d -> b 1 d', b=x.shape[0])
        return torch.cat([cls, tokens], dim=1)

demo_embed = PatchEmbedding(patch_dim=PATCH_DIM, dim=128)
demo_out   = demo_embed(torch.randn(4, 1, 28, 28))
print(f'塊嵌入輸出 / Output: {demo_out.shape}  (batch=4, 16塊+1CLS, dim=128)')

---
## 第四步：位置編碼 / Step 4 — Positional Encoding

**中文：**  
Transformer 沒有內建順序感。若沒有位置編碼，左上角的塊和右下角的塊對模型來說完全相同。  
我們為每個詞元加上**可學習的位置向量**：
$$x_i = x_i + \text{pos\_embed}[i]$$

**EN:**  
Without positional encoding, patch #0 (top-left) looks identical to patch #15 (bottom-right).  
We add a learnable positional vector to each token.

In [ ]:
num_tokens = NUM_PATCHES + 1
pos_embedding_demo = nn.Parameter(torch.randn(1, num_tokens, 64))
pe = pos_embedding_demo.squeeze(0).detach()
pe_norm = F.normalize(pe, dim=-1)
sim = (pe_norm @ pe_norm.T).numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im = axes[0].imshow(sim, cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_title('位置編碼餘弦相似度（訓練前隨機）\nPos. embedding similarity (before training)', fontsize=10)
axes[0].set_xlabel('詞元索引 Token index (0=CLS, 1-16=patches)')
axes[0].set_ylabel('詞元索引 Token index')
plt.colorbar(im, ax=axes[0])

grid = np.arange(NUM_PATCHES).reshape(4, 4)
axes[1].imshow(grid, cmap='viridis')
for i in range(4):
    for j in range(4):
        axes[1].text(j, i, f'P{grid[i,j]}', ha='center', va='center',
                     color='white', fontsize=11, fontweight='bold')
axes[1].set_title('訓練後期望：相鄰塊的位置編碼更相似\nAfter training: adjacent patches → similar embeddings', fontsize=10)
axes[1].axis('off')
plt.tight_layout()
plt.show()

---
## 第五步：多頭自注意力 / Step 5 — Multi-Head Self-Attention

**中文：**  
每個詞元生成三個向量：
- **Q（查詢）**：我在找什麼？
- **K（鍵）**：我包含什麼？
- **V（值）**：若被選中，我輸出什麼？

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**多頭**：並行執行 `h` 個注意力，每頭專注不同特徵，最後拼接。

**EN:** Each token generates Q/K/V. Scaled dot-product attention, run in parallel across `h` heads.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, dim, heads, dropout=0.0):
        super().__init__()
        assert dim % heads == 0
        self.heads    = heads
        self.dim_head = dim // heads
        self.scale    = self.dim_head ** -0.5
        self.to_qkv   = nn.Linear(dim, dim * 3, bias=False)
        self.to_out   = nn.Sequential(nn.Linear(dim, dim), nn.Dropout(dropout))

    def forward(self, x):
        b, n, _ = x.shape
        h = self.heads
        qkv = self.to_qkv(x).chunk(3, dim=-1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h=h), qkv)
        dots = torch.einsum('b h i d, b h j d -> b h i j', q, k) * self.scale
        attn = dots.softmax(dim=-1)
        out  = torch.einsum('b h i j, b h j d -> b h i d', attn, v)
        out  = rearrange(out, 'b h n d -> b n (h d)')
        return self.to_out(out), attn

mha = MultiHeadAttention(dim=128, heads=4)
x   = torch.randn(2, 17, 128)
out, attn_w = mha(x)
print(f'輸入 Input   : {x.shape}')
print(f'輸出 Output  : {out.shape}')
print(f'注意力矩陣   : {attn_w.shape}  (batch, 頭數, 詞元數, 詞元數)')

---
## 第六步：Transformer 塊 / Step 6 — Transformer Block

**中文：**  
每個 Transformer 塊有兩個子層，各自帶殘差連接與 Pre-Norm：
1. **多頭自注意力**——詞元互相通訊
2. **前饋網路（FFN, GELU）**——每個詞元獨立非線性變換

**EN:** MHA + FFN, each with Pre-LayerNorm and residual connection.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, dim, mlp_dim, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, dim),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, dim, heads, mlp_dim, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = MultiHeadAttention(dim, heads, dropout)
        self.ff    = FeedForward(dim, mlp_dim, dropout)

    def forward(self, x):
        attn_out, attn_weights = self.attn(self.norm1(x))
        x = x + attn_out
        x = x + self.ff(x)
        return x, attn_weights


block = TransformerBlock(dim=128, heads=4, mlp_dim=256)
x_in  = torch.randn(2, 17, 128)
x_out, _ = block(x_in)
print(f'輸入 Input : {x_in.shape}')
print(f'輸出 Output: {x_out.shape}  （殘差保形，形狀不變）')

---
## 第七步：完整 MNIST 分類模型 / Step 7 — Full MNIST Classification Model

**中文：**  
將所有元件整合為一個完整的 `MNISTViT` 分類器。這是這個架構在視覺領域最基礎的應用：給定一張 28×28 的手寫數字圖，輸出它屬於 0–9 哪個類別。

```
輸入圖像 (batch, 1, 28, 28)
    ↓
PatchEmbedding  → [CLS, P0…P15]  (batch, 17, dim=128)
    ↓  + 位置編碼
TransformerBlock × 6             （每層：注意力 + FFN + 殘差）
    ↓
取 CLS 詞元  (batch, 128)
    ↓
LayerNorm → Linear(128, 10) → logits
```

**EN:** All components assembled into a clean `MNISTViT` classifier. Input: 28×28 grayscale. Output: digit class 0–9.

In [ ]:
class MNISTViT(nn.Module):
    """
    基於 Transformer 架構的 MNIST 手寫數字分類器。
    Vision Transformer classifier for MNIST handwritten digits.

    架構 Architecture:
        image_size=28, patch_size=7  → 16 patches
        dim=128, depth=6, heads=4, mlp_dim=256
        Output: 10-class logits
    """
    def __init__(
        self,
        image_size: int = 28,
        patch_size: int = 7,
        num_classes: int = 10,
        dim: int = 128,
        depth: int = 6,
        heads: int = 4,
        mlp_dim: int = 256,
        dropout: float = 0.1,
    ):
        super().__init__()
        assert image_size % patch_size == 0, '圖像尺寸必須能被塊尺寸整除'

        num_patches = (image_size // patch_size) ** 2   # 16
        patch_dim   = patch_size * patch_size            # 49 (grayscale)

        # ── 1. 塊嵌入層 Patch embedding ──────────────────────────
        self.patch_embed = PatchEmbedding(patch_dim, dim)

        # ── 2. 可學習位置編碼 Learnable positional encoding ───────
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, dim))
        self.dropout   = nn.Dropout(dropout)

        # ── 3. Transformer 編碼器 Encoder ─────────────────────────
        self.encoder = nn.ModuleList([
            TransformerBlock(dim, heads, mlp_dim, dropout)
            for _ in range(depth)
        ])

        # ── 4. 分類頭 Classification head ─────────────────────────
        self.classifier = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, num_classes)
        )

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        # 步驟 1：塊嵌入 + 位置編碼
        x = self.patch_embed(x)
        x = self.dropout(x + self.pos_embed)

        # 步驟 2：通過所有 Transformer 塊
        attn_maps = []
        for block in self.encoder:
            x, attn = block(x)
            attn_maps.append(attn)

        # 步驟 3：取 CLS 詞元，分類
        logits = self.classifier(x[:, 0])

        return (logits, attn_maps) if return_attn else logits


# ── 建立模型 / Instantiate model ────────────────────────────────
model = MNISTViT(
    image_size=28, patch_size=7, num_classes=10,
    dim=128, depth=6, heads=4, mlp_dim=256, dropout=0.1
).to(device)

# 參數統計 / Parameter count
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'總參數量   Total parameters    : {total:,}')
print(f'可訓練參數 Trainable parameters: {trainable:,}')

# 前向傳播測試 / Forward pass sanity check
dummy  = torch.randn(4, 1, 28, 28).to(device)
logits = model(dummy)
print(f'輸出形狀   Output shape        : {logits.shape}  ✓')

### 模型架構圖 / Model Architecture Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
ax.set_xlim(0, 16)
ax.set_ylim(0, 5)
ax.axis('off')

boxes = [
    (0.2,  1.5, 1.5, 2.0, '#AED6F1', '輸入\n(1,28,28)'),
    (2.1,  1.2, 2.2, 2.6, '#A9DFBF', '塊嵌入\nPatch Embed\n49→128\n+CLS→17tok'),
    (4.7,  1.2, 2.2, 2.6, '#F9E79F', '位置編碼\nPos Embed\n+learnable\n(1,17,128)'),
    (7.3,  0.8, 3.2, 3.4, '#FAD7A0', 'Transformer Encoder\n×6 塊 blocks\n  MHA: 4頭 heads\n  FFN: 128→256→128\n  殘差+LayerNorm'),
    (10.9, 1.2, 2.2, 2.6, '#D2B4DE', 'CLS 詞元\nCLS Token\nx[:,0]\n(batch,128)'),
    (13.5, 1.4, 2.2, 2.2, '#F1948A', '分類頭\nClassifier\nLN+Linear\n→10類'),
]

for (x, y, w, h, color, label) in boxes:
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.12',
                                    facecolor=color, edgecolor='#555', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center', fontsize=8, fontweight='bold')

arrow_segs = [(1.7, 2.5), (4.3, 4.7), (6.9, 7.3), (10.5, 10.9), (13.1, 13.5)]
for (x1, x2) in arrow_segs:
    ax.annotate('', xy=(x2, 2.5), xytext=(x1, 2.5),
                arrowprops=dict(arrowstyle='->', color='#333', lw=2))

ax.set_title('MNISTViT 完整架構 / MNISTViT Full Architecture', fontsize=13, pad=8)
plt.tight_layout()
plt.show()

---
## 第八步：訓練與評估 / Step 8 — Training & Evaluation

**中文：** Adam 最佳化器 + OneCycleLR 學習率排程，訓練 10 個 epoch。  
**EN:** Adam + OneCycleLR, 10 epochs.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-3,
    epochs=10, steps_per_epoch=len(train_loader)
)
criterion = nn.CrossEntropyLoss()


def evaluate(mdl, loader):
    mdl.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = mdl(imgs).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return correct / total * 100


train_losses, test_accs = [], []
EPOCHS = 10

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    acc      = evaluate(model, test_loader)
    train_losses.append(avg_loss)
    test_accs.append(acc)
    print(f'Epoch {epoch:2d}/{EPOCHS} | 損失 Loss: {avg_loss:.4f} | 測試準確率 Test Acc: {acc:.2f}%')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(range(1, EPOCHS+1), train_losses, marker='o', color='steelblue', linewidth=2)
ax1.set_title('訓練損失 / Training Loss', fontsize=12)
ax1.set_xlabel('訓練輪次 Epoch')
ax1.set_ylabel('交叉熵損失 Cross-Entropy Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, EPOCHS+1), test_accs, marker='o', color='forestgreen', linewidth=2)
ax2.set_title('測試準確率 / Test Accuracy', fontsize=12)
ax2.set_xlabel('訓練輪次 Epoch')
ax2.set_ylabel('準確率 Accuracy (%)')
ax2.set_ylim(90, 100)
ax2.grid(True, alpha=0.3)

plt.suptitle(f'MNISTViT 訓練結果 / Training Results — Final Acc: {test_accs[-1]:.2f}%', fontsize=13)
plt.tight_layout()
plt.show()

---
## 第九步：混淆矩陣與各類準確率 / Step 9 — Confusion Matrix & Per-class Accuracy

**中文：**  
混淆矩陣顯示每個數字被分類為哪些類別。對角線是正確預測，非對角線是錯誤。  
藉此可以看出模型在哪些數字上容易混淆（如 4 和 9、3 和 8）。

**EN:**  
The confusion matrix shows which digits get misclassified as which. Diagonal = correct. Off-diagonal = errors.  
Reveals which digit pairs confuse the model (e.g., 4 vs 9, 3 vs 8).

In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        preds = model(imgs).argmax(dim=1).cpu()
        all_preds.append(preds)
        all_labels.append(labels)

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

cm = confusion_matrix(all_labels, all_preds)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── 混淆矩陣 / Confusion matrix ───────────────────────────────
im = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
axes[0].set_title('混淆矩陣 / Confusion Matrix\n（列=真實標籤 row=true, 行=預測 col=predicted）', fontsize=11)
plt.colorbar(im, ax=axes[0])
tick_marks = np.arange(10)
axes[0].set_xticks(tick_marks); axes[0].set_yticks(tick_marks)
axes[0].set_xticklabels([str(i) for i in range(10)])
axes[0].set_yticklabels([str(i) for i in range(10)])
axes[0].set_xlabel('預測標籤 Predicted label', fontsize=10)
axes[0].set_ylabel('真實標籤 True label', fontsize=10)

thresh = cm.max() / 2.0
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    axes[0].text(j, i, format(cm[i, j], 'd'),
                 ha='center', va='center', fontsize=8,
                 color='white' if cm[i, j] > thresh else 'black')

# ── 各類準確率長條圖 / Per-class accuracy bar chart ──────────
per_class_acc = cm.diagonal() / cm.sum(axis=1) * 100
bar_colors = ['#E74C3C' if a < 98 else '#2ECC71' for a in per_class_acc]
bars = axes[1].bar(range(10), per_class_acc, color=bar_colors, edgecolor='white', linewidth=1.2)
axes[1].set_xticks(range(10))
axes[1].set_xticklabels([f'數字 {i}' for i in range(10)], rotation=30, ha='right')
axes[1].set_ylim(94, 100.5)
axes[1].set_title('各類別準確率 / Per-class Accuracy', fontsize=11)
axes[1].set_ylabel('準確率 Accuracy (%)')
axes[1].axhline(y=np.mean(per_class_acc), color='navy', linestyle='--',
                linewidth=1.5, label=f'平均 Mean: {np.mean(per_class_acc):.2f}%')
axes[1].legend(fontsize=9)
axes[1].grid(axis='y', alpha=0.3)

for bar, acc in zip(bars, per_class_acc):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{acc:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.suptitle(f'MNISTViT 測試集評估 / Test Set Evaluation — Overall: {test_accs[-1]:.2f}%', fontsize=12)
plt.tight_layout()
plt.show()

print('\n── 各類別準確率 / Per-class Accuracy ──')
for i, acc in enumerate(per_class_acc):
    bar = '█' * int(acc / 2) + '░' * (50 - int(acc / 2))
    print(f'數字 {i}: {acc:.2f}%  {bar}')

### 範例預測（含信心分數）/ Sample Predictions with Confidence

In [ ]:
model.eval()

# 取一批測試圖像 / Get a batch of test images
test_iter = iter(test_loader)
imgs_batch, labels_batch = next(test_iter)
imgs_batch = imgs_batch.to(device)

with torch.no_grad():
    logits_batch = model(imgs_batch)
    probs_batch  = F.softmax(logits_batch, dim=-1)
    preds_batch  = probs_batch.argmax(dim=1)

# 顯示前 20 張（正確綠框，錯誤紅框）/ Show first 20
fig, axes = plt.subplots(4, 5, figsize=(14, 11))
for idx, ax in enumerate(axes.flat):
    img   = imgs_batch[idx].cpu().squeeze().numpy()
    true  = labels_batch[idx].item()
    pred  = preds_batch[idx].item()
    conf  = probs_batch[idx, pred].item() * 100
    correct = (pred == true)

    ax.imshow(img, cmap='gray')
    ax.set_title(
        f'真 True: {true}\n預 Pred: {pred}  ({conf:.1f}%)',
        fontsize=9,
        color='#1E8449' if correct else '#C0392B',
        fontweight='bold'
    )
    # 邊框顏色顯示正確/錯誤 / Border color = correct/wrong
    for spine in ax.spines.values():
        spine.set_edgecolor('#1E8449' if correct else '#C0392B')
        spine.set_linewidth(3)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('預測範例：綠框=正確，紅框=錯誤，括號內為信心分數\n'
             'Predictions: green=correct, red=wrong, (confidence%)',
             fontsize=12)
plt.tight_layout()
plt.show()

---
## 第十步：注意力視覺化 / Step 10 — Attention Visualization

**中文：**  
訓練完成後觀察模型在看哪裡。CLS 詞元對各塊的注意力揭示哪些像素區域對分類最重要。

**EN:**  
CLS token attention to each patch reveals which regions were most informative for classification.

In [ ]:
model.eval()

class_images = {}
for img, label in test_data:
    lbl = label if isinstance(label, int) else label.item()
    if lbl not in class_images:
        class_images[lbl] = img
    if len(class_images) == 10:
        break

fig, axes = plt.subplots(3, 10, figsize=(20, 7))

for col, digit in enumerate(range(10)):
    img = class_images[digit].unsqueeze(0).to(device)

    with torch.no_grad():
        logits, attn_maps = model(img, return_attn=True)
        pred = logits.argmax().item()
        conf = F.softmax(logits, dim=-1)[0, pred].item() * 100

    axes[0, col].imshow(img.cpu().squeeze(), cmap='gray')
    axes[0, col].set_title(
        f'真:{digit} 預:{pred}\n{conf:.1f}%', fontsize=8,
        color='#1E8449' if pred == digit else '#C0392B'
    )
    axes[0, col].axis('off')

    last_attn = attn_maps[-1][0]
    cls_attn  = last_attn[:, 0, 1:].mean(dim=0).cpu().numpy().reshape(4, 4)
    cls_attn  = (cls_attn - cls_attn.min()) / (cls_attn.max() - cls_attn.min() + 1e-8)

    axes[1, col].imshow(cls_attn, cmap='hot', vmin=0, vmax=1)
    axes[1, col].set_title(f'數字 {digit}', fontsize=8)
    axes[1, col].axis('off')

    attn_up = np.kron(cls_attn, np.ones((7, 7)))
    axes[2, col].imshow(img.cpu().squeeze().numpy(), cmap='gray', alpha=0.45)
    axes[2, col].imshow(attn_up, cmap='hot', alpha=0.65)
    axes[2, col].axis('off')

axes[0, 0].set_ylabel('原圖', fontsize=8)
axes[1, 0].set_ylabel('注意力圖', fontsize=8)
axes[2, 0].set_ylabel('疊加圖', fontsize=8)
plt.suptitle('CLS 詞元注意力（最後一層，多頭平均）/ CLS Token Attention — Last Layer Mean', fontsize=12)
plt.tight_layout()
plt.show()

---
## 第十一步：注意力傳播 / Step 11 — Attention Rollout

**中文：**  
**注意力傳播**將所有層的注意力矩陣連乘，加入單位矩陣模擬殘差，得到更乾淨的全域歸因：
$$\text{rollout} = A_1 \cdot A_2 \cdots A_L$$

**EN:** Multiply attention matrices across all layers (with residual identity) for cleaner global attribution.

In [ ]:
def attention_rollout(attn_maps):
    """跨所有層的注意力傳播 / Propagate attention across all layers."""
    num_tokens = attn_maps[0].shape[-1]
    rollout    = torch.eye(num_tokens).to(attn_maps[0].device)
    for attn in attn_maps:
        attn_avg = attn[0].mean(dim=0)
        attn_avg = attn_avg + torch.eye(num_tokens, device=attn_avg.device)
        attn_avg = attn_avg / attn_avg.sum(dim=-1, keepdim=True)
        rollout  = attn_avg @ rollout
    return rollout


fig, axes = plt.subplots(2, 10, figsize=(20, 5))

for col, digit in enumerate(range(10)):
    img = class_images[digit].unsqueeze(0).to(device)
    with torch.no_grad():
        _, attn_maps = model(img, return_attn=True)

    rollout = attention_rollout(attn_maps)
    cls_r   = rollout[0, 1:].cpu().numpy().reshape(4, 4)
    cls_r   = (cls_r - cls_r.min()) / (cls_r.max() - cls_r.min() + 1e-8)

    axes[0, col].imshow(cls_r, cmap='plasma', vmin=0, vmax=1)
    axes[0, col].set_title(f'數字 {digit}', fontsize=9)
    axes[0, col].axis('off')

    attn_up = np.kron(cls_r, np.ones((7, 7)))
    axes[1, col].imshow(img.cpu().squeeze().numpy(), cmap='gray', alpha=0.4)
    axes[1, col].imshow(attn_up, cmap='plasma', alpha=0.7)
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('傳播圖', fontsize=8)
axes[1, 0].set_ylabel('疊加圖', fontsize=8)
plt.suptitle('注意力傳播圖 / Attention Rollout — Propagated Across All Transformer Layers', fontsize=12)
plt.tight_layout()
plt.show()

---
## 第十二步：與 vit-pytorch 比較 / Step 12 — Compare with vit-pytorch

**中文：** 以相同超參數訓練 `vit-pytorch` 的 `SimpleViT`，比較兩者的準確率曲線。  
**EN:** Same hyperparameters, compare accuracy curves between our model and vit-pytorch.

In [ ]:
from vit_pytorch import SimpleViT

vit_lib = SimpleViT(
    image_size=28, patch_size=7, num_classes=10,
    dim=128, depth=6, heads=4, mlp_dim=256, channels=1
).to(device)

opt2 = torch.optim.Adam(vit_lib.parameters(), lr=3e-4)
sch2 = torch.optim.lr_scheduler.OneCycleLR(
    opt2, max_lr=3e-3, epochs=10, steps_per_epoch=len(train_loader)
)

lib_accs = []
for epoch in range(1, 11):
    vit_lib.train()
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        opt2.zero_grad()
        criterion(vit_lib(imgs), labels).backward()
        opt2.step()
        sch2.step()
    acc = evaluate(vit_lib, test_loader)
    lib_accs.append(acc)
    print(f'Epoch {epoch:2d}/10 | SimpleViT Test Acc: {acc:.2f}%')

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(range(1, 11), test_accs, marker='o', linewidth=2,
         color='steelblue',  label=f'MNISTViT（本筆記）  最終: {test_accs[-1]:.2f}%')
plt.plot(range(1, 11), lib_accs, marker='s', linewidth=2,
         color='darkorange', label=f'vit-pytorch SimpleViT  最終: {lib_accs[-1]:.2f}%')
plt.xlabel('訓練輪次 Epoch')
plt.ylabel('測試準確率 Test Accuracy (%)')
plt.title('MNISTViT vs. vit-pytorch SimpleViT 在 MNIST 上的比較')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(90, 100)
plt.tight_layout()
plt.show()

---
## 總結 / Summary

### ViT 在視覺領域的應用路徑 / ViT Application Path in Vision Domain

```
本筆記的任務（圖像分類）
This notebook's task (Image Classification)
        ↓ 換資料集 swap dataset
CIFAR-10 / 醫療X光 / 衛星圖像
        ↓ 換輸出頭 swap output head
目標偵測（Detection）/ 語意分割（Segmentation）
        ↓ 加時間維度 add temporal dimension
影片理解（Video Understanding）with ViT-3D
        ↓ 跨模態 cross-modal
多模態（Multimodal）CLIP / BLIP / GPT-4V
```

### 各元件作用 / What Each Component Learns

| 元件 Component | 學到什麼 What it learns |
|---|---|
| 塊嵌入 Patch embedding | 每塊的局部紋理與形狀的緊湊表示 |
| 位置編碼 Positional embedding | 每塊在圖像中的空間位置 |
| 自注意力 Self-attention | 哪些塊之間存在關聯（全域感受野）|
| 前饋網路 FFN | 每個詞元的非線性特徵變換 |
| CLS 詞元 CLS token | 全圖摘要——透過注意力匯聚所有塊的資訊 |

### 為何 ViT 在 MNIST 上有效 / Why ViT Works on MNIST

> **中文：** 即使只有 16 個塊，全域注意力讓 CLS 詞元從第一層就能看到整個數字的筆畫結構——這在 CNN 中只有最深層才能辦到。  
> **EN:** Even with only 16 patches, global attention lets the CLS token immediately see the full stroke pattern — something a CNN achieves only in its deepest layers.